In [4]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont

import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import visual_identity_solver
import visual_cleaning

importlib.reload(visual_identity_solver)
importlib.reload(visual_cleaning)

solver = visual_identity_solver

from visual_cleaning import clean_visual_dataframe, add_clean_emotion_columns

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DEBATE_NAME = "Seguro_vs_Martins_December_6"

print("Project root:", PROJECT_ROOT)
print("Debate:", DEBATE_NAME)
print("Loaded solver from:", visual_identity_solver.__file__)
print("Loaded cleaning from:", visual_cleaning.__file__)

Project root: C:\Users\lucas\Documents\big_data
Debate: Seguro_vs_Martins_December_6
Loaded solver from: C:\Users\lucas\Documents\big_data\src\visual_identity_solver.py
Loaded cleaning from: C:\Users\lucas\Documents\big_data\src\visual_cleaning.py


In [5]:
visual_candidates = list(DATA_DIR.rglob(f"*{DEBATE_NAME}*visual*.pkl"))

if len(visual_candidates) == 0:
    print("Available visual pickle files:")
    for path in DATA_DIR.rglob("*visual*.pkl"):
        print(path)

    raise FileNotFoundError(f"No visual pickle found for {DEBATE_NAME}")

VISUAL_PKL = visual_candidates[0]

print("Using visual pickle:")
print(VISUAL_PKL)

Using visual pickle:
C:\Users\lucas\Documents\big_data\data\raw_features\Seguro_vs_Martins_December_6_visual.pkl


In [6]:
solver_output_path = PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_SINGLE_UNCONSTRAINED_CLEANED_predictions.pkl"
solver_frames_path = PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_SINGLE_UNCONSTRAINED_CLEANED_frames.pkl"

FORCE_RERUN = True

# Since you installed the libraries, try True.
# If InsightFace fails or coverage is low, the solver falls back automatically.
USE_INSIGHTFACE = True

if solver_output_path.exists() and solver_frames_path.exists() and not FORCE_RERUN:
    print("Loading cached single-unconstrained solver results...")
    out = pd.read_pickle(solver_output_path)
    df_solver = pd.read_pickle(solver_frames_path)
    model_name = "cached_single_unconstrained"

else:
    cfg = solver.Config(
        pkl=VISUAL_PKL,
        project_root=PROJECT_ROOT,
        frames_root=Path("Frames"),
        out=PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_single_unconstrained_outputs",

        use_insightface=USE_INSIGHTFACE,

        # Multi-person constraints stay active
        force_small_two_has_person2=True,

        # IMPORTANT:
        # Do not use early single shots as person_2 anchors.
        # Single-person shots should be model/prototype predictions only.
        use_first_single_p2_prior=False,
        early_single_p2_always=False,

        constraint_first=True,

        prototype_temperature=2.25,
        low_confidence_threshold=0.55,

        two_large_sum_area=0.55,
        two_touching_gap=0.035,

        save_debug_images=False,
        annotate_every=0,
    )

    print("Loading raw pickle...")
    df_raw = solver.load_visual_pickle(VISUAL_PKL)

    print("Cleaning visual detections first...")
    df_clean = clean_visual_dataframe(df_raw)
    df_clean = add_clean_emotion_columns(df_clean)

    # IMPORTANT:
    # The solver expects columns named Poses and Fer.
    # We replace them with the cleaned detections.
    df_solver = df_clean.copy()
    df_solver["Poses"] = df_solver["Clean_Poses"]
    df_solver["Fer"] = df_solver["Clean_Fer"]

    df_solver["frame_num"] = df_solver["Frame"].map(solver.parse_frame_num)
    df_solver = df_solver.sort_values("frame_num").reset_index(drop=True)

    df_solver["n_poses"] = df_solver["Poses"].map(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    df_solver["n_faces"] = df_solver["Fer"].map(
        lambda x: sum(1 for f in x if isinstance(f, dict)) if isinstance(x, list) else 0
    )

    print("Cleaned frame composition:")
    display(pd.crosstab(df_solver["n_poses"], df_solver["n_faces"]))

    print("Inferring frame size...")
    width, height = solver.infer_frame_size(df_solver, PROJECT_ROOT)
    print("Frame size:", width, height)

    print("Building detection table from CLEANED boxes...")
    det = solver.build_detection_table(df_solver, cfg, width, height)
    print("Cleaned detections:", len(det))

    print("Creating base features...")
    base_X, base_names, aux = solver.create_base_features(det, width, height)
    det = pd.concat([det.reset_index(drop=True), aux.reset_index(drop=True)], axis=1)

    print("Extracting InsightFace embeddings if available...")
    arc_X = solver.extract_insightface_embeddings(det, cfg)

    print("Building model features...")
    X = solver.build_model_features(det, base_X, arc_X, cfg)

    print("Assigning weak labels...")
    det = solver.assign_weak_labels(det, cfg)

    print("Weak anchor counts:")
    display(
        det.loc[det["weak_label"] >= 0, "weak_label"]
        .map(solver.INT_TO_LABEL)
        .value_counts()
    )

    print("Fitting identity model...")
    if cfg.constraint_first:
        probs, pred, model_name = solver.fit_prototype_identity_model(X, det, cfg)
    else:
        probs, pred, model_name = solver.fit_identity_model(X, det, cfg)

    print("Model:", model_name)

    print("Applying frame constraints...")
    out = solver.apply_frame_constraints(det, probs, cfg)

    # ------------------------------------------------------------
    # IMPORTANT CORRECTION:
    # Do NOT smooth single-person runs.
    # Do NOT apply final frame constraints to single-person shots.
    # Single-person shots should stay as raw model/prototype predictions.
    # ------------------------------------------------------------

    out["unconstrained_model_person"] = out["model_person"]
    out["unconstrained_model_confidence"] = out["model_confidence"]

    single_mask = out["n_faces"] == 1

    out.loc[single_mask, "person_label"] = out.loc[single_mask, "unconstrained_model_person"]
    out.loc[single_mask, "confidence"] = out.loc[single_mask, "unconstrained_model_confidence"]
    out.loc[single_mask, "assignment_source"] = "model_single_unconstrained"

    # For widget display, make model_person match the final chosen label.
    # The original model output is preserved in unconstrained_model_person.
    out["model_person"] = out["person_label"]
    out["model_confidence"] = out["confidence"]

    out.to_pickle(solver_output_path)
    df_solver.to_pickle(solver_frames_path)

    print("Saved:")
    print(solver_output_path)
    print(solver_frames_path)

print("Final assignments:")
display(out["person_label"].value_counts())

print("Assignment sources:")
display(out["assignment_source"].value_counts())

print("Person 2 detections by source:")
display(
    out[out["person_label"] == "person_2"]
    ["assignment_source"]
    .value_counts()
)

Loading raw pickle...
Cleaning visual detections first...
Cleaned frame composition:


n_faces,1,2,3
n_poses,,,
0,2,1,3
1,1105,18,12
2,0,765,63
3,0,2,95


Inferring frame size...
Frame size: 1280 720
Building detection table from CLEANED boxes...
Cleaned detections: 3198
Creating base features...
Extracting InsightFace embeddings if available...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\lucas/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\lucas/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\lucas/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\lucas/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Appli

InsightFace embeddings:   0%|          | 0/2066 [00:00<?, ?it/s]c:\Python\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
InsightFace embeddings: 100%|██████████| 2066/2066 [12:03<00:00,  2.86it/s]


InsightFace embedding coverage: 100.0%
Building model features...
Assigning weak labels...
Weak anchor counts:


...

person_label
person_A    1487
person_C    1443
person_B     268
Name: count, dtype: int64

Assignment sources:


assignment_source
hard_rule_2_large_or_touching_left_right    1476
model_single_unconstrained                  1107
hard_rule_3_faces_left_to_right              519
soft_rule_2_small_one_is_person_B             96
Name: count, dtype: int64

Person 2 detections by source:


Series([], Name: count, dtype: int64)

In [7]:
def resolve_frame_path_for_widget(frame_value):
    path = solver.resolve_frame_path(
        PROJECT_ROOT,
        Path("Frames"),
        frame_value,
    )

    if path is not None and path.exists():
        return path

    return None


def get_font(size=18):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()


def draw_text(draw, x, y, text, fill=(255, 255, 0), font_size=18):
    font = get_font(font_size)
    box = draw.textbbox((x, y), text, font=font)

    draw.rectangle(
        [box[0] - 4, box[1] - 4, box[2] + 4, box[3] + 4],
        fill=(0, 0, 0),
    )

    draw.text(
        (x, y),
        text,
        fill=fill,
        font=font,
    )


def load_widget_frame(frame_value):
    path = resolve_frame_path_for_widget(frame_value)

    if path is None:
        img = Image.new("RGB", (1280, 720), color=(30, 30, 30))
        return img, None

    img = Image.open(path).convert("RGB")
    return img, path


def draw_box(draw, bbox, outline, width=3):
    if bbox is None:
        return

    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
    except Exception:
        return

    if not np.isfinite([x1, y1, x2, y2]).all():
        return

    draw.rectangle([x1, y1, x2, y2], outline=outline, width=width)


def get_frame_row(frame_value):
    subset = df_solver[df_solver["Frame"] == frame_value]

    if len(subset) == 0:
        return None

    return subset.iloc[0]


def annotate_solver_frame(
    frame_value,
    show_face_boxes=True,
    show_body_boxes=True,
    show_identity=True,
):
    img, frame_path = load_widget_frame(frame_value)
    draw = ImageDraw.Draw(img)

    frame_row = get_frame_row(frame_value)
    frame_dets = out[out["frame"] == frame_value].copy()
    frame_dets = frame_dets.sort_values("face_cx")

    # Blue = cleaned body boxes
    if show_body_boxes and frame_row is not None:
        poses = frame_row["Poses"] if isinstance(frame_row["Poses"], list) else []

        for i, pose in enumerate(poses):
            if not isinstance(pose, dict) or "bbox" not in pose:
                continue

            bbox = pose["bbox"]

            draw_box(draw, bbox, outline=(0, 150, 255), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                max(0, y1 - 22),
                f"body_{i}",
                fill=(0, 180, 255),
                font_size=13,
            )

    # Green = cleaned face boxes
    if show_face_boxes and frame_row is not None:
        faces = frame_row["Fer"] if isinstance(frame_row["Fer"], list) else []

        for i, face in enumerate(faces):
            if not isinstance(face, dict) or "bbox" not in face:
                continue

            bbox = face["bbox"]

            draw_box(draw, bbox, outline=(0, 255, 0), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                y2 + 4,
                f"face_{i}",
                fill=(0, 255, 0),
                font_size=13,
            )

    # Yellow = final identity around solver face detection
    if show_identity:
        for _, r in frame_dets.iterrows():
            bbox = [r["face_x1"], r["face_y1"], r["face_x2"], r["face_y2"]]

            label = str(r["person_label"])
            conf = float(r["confidence"])

            label_text = f"{label} {conf:.2f}"

            draw_box(draw, bbox, outline=(255, 255, 0), width=5)

            x1, y1, x2, y2 = bbox

            draw_text(
                draw,
                x1,
                max(0, y1 - 46),
                label_text,
                fill=(255, 255, 0),
                font_size=20,
            )

    if len(frame_dets) > 0:
        first = frame_dets.iloc[0]
        info = (
            f"frame={first['frame_num']} | "
            f"faces={first['n_faces']} | "
            f"poses={first['n_poses']} | "
            f"source={first['assignment_source']}"
        )
    elif frame_row is not None:
        info = (
            f"frame={frame_row['frame_num']} | "
            f"faces={frame_row['n_faces']} | "
            f"poses={frame_row['n_poses']} | "
            f"No solver detections"
        )
    else:
        info = "No detections"

    draw_text(
        draw,
        10,
        img.size[1] - 35,
        info,
        fill=(255, 255, 255),
        font_size=15,
    )

    return img, frame_path

In [8]:
frame_table = (
    df_solver[["Frame", "frame_num", "n_faces", "n_poses"]]
    .drop_duplicates()
    .sort_values("frame_num")
    .reset_index(drop=True)
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_table) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

show_body_checkbox = widgets.Checkbox(
    value=True,
    description="Body boxes",
)

show_face_checkbox = widgets.Checkbox(
    value=True,
    description="Face boxes",
)

show_identity_checkbox = widgets.Checkbox(
    value=True,
    description="Identity",
)

jump_frame = widgets.IntText(
    value=0,
    description="Go to frame:",
)

jump_button = widgets.Button(
    description="Jump",
    button_style="info",
)

output = widgets.Output()


def jump_to_frame(_):
    target = int(jump_frame.value)

    idx = (frame_table["frame_num"] - target).abs().idxmin()

    frame_slider.value = int(idx)


jump_button.on_click(jump_to_frame)


def update_frame(change=None):
    idx = frame_slider.value
    row = frame_table.iloc[idx]

    frame_value = row["Frame"]

    with output:
        clear_output(wait=True)

        img, frame_path = annotate_solver_frame(
            frame_value,
            show_face_boxes=show_face_checkbox.value,
            show_body_boxes=show_body_checkbox.value,
            show_identity=show_identity_checkbox.value,
        )

        plt.figure(figsize=(13, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

        frame_dets = out[out["frame"] == frame_value].sort_values("face_cx")

        print("slider index:", idx)
        print("frame_num:", row["frame_num"])
        print("frame:", frame_value)
        print("frame path:", frame_path)
        print("cleaned faces:", row["n_faces"])
        print("cleaned bodies:", row["n_poses"])
        print("solver detections in frame:", len(frame_dets))

        display_cols = [
            "face_idx_lr",
            "person_label",
            "confidence",
            "assignment_source",
            "model_person",
            "model_confidence",
            "unconstrained_model_person",
            "unconstrained_model_confidence",
            "weak_source",
        ]

        display_cols = [c for c in display_cols if c in frame_dets.columns]

        if len(frame_dets) > 0:
            display(frame_dets[display_cols])


frame_slider.observe(update_frame, names="value")
show_body_checkbox.observe(update_frame, names="value")
show_face_checkbox.observe(update_frame, names="value")
show_identity_checkbox.observe(update_frame, names="value")

display(
    widgets.VBox(
        [
            frame_slider,
            widgets.HBox(
                [
                    show_body_checkbox,
                    show_face_checkbox,
                    show_identity_checkbox,
                ]
            ),
            widgets.HBox([jump_frame, jump_button]),
            output,
        ]
    )
)

update_frame()